# MuSGD × YOLO26 journal development: campaign

Private research execution. Notebook cap: **42000 seconds**, including installation and preparation. Training uses one Kaggle T4. No training runs on the user's laptop.

The frozen plan is embedded with inspectable source and tests. Primary comparisons use 1,280 updates and 20,480 image exposures for every condition. Validation-only tuning uses 10 exposure blocks; settings transfer to 20-block confirmation. Independent paired blocks vary training/subset/annotation seeds jointly. VOC2007 official splits are converted with all difficult objects included; scores are not official VOC AP.

Test metrics are opened only after the complete mandatory confirmation grid and a checkpoint freeze. Failures and budget shortfalls remain visible. A completed notebook is not a publication or novelty claim.


In [ ]:
%pip install -q "ultralytics==8.4.187" "PyYAML==6.0.3" "matplotlib>=3.8,<4" "Pillow>=10" "pytest>=8,<10"

In [ ]:
from pathlib import Path
import base64, hashlib, io, json, os, subprocess, sys, traceback, zipfile
WORKING=Path('/kaggle/working')
PROJECT=WORKING/'journal_source'
PROJECT.mkdir(parents=True,exist_ok=True)
bundle=base64.b64decode('...')
assert hashlib.sha256(bundle).hexdigest()=='2fbb47c3e39869cf8668f5f7cd77849a4e40222cb3f7cb585c9fba6b2d817848'
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    assert all((PROJECT/member).resolve().is_relative_to(PROJECT.resolve()) for member in archive.namelist())
    archive.extractall(PROJECT)
sys.path.insert(0,str(PROJECT))
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
subprocess.run([sys.executable,'-m','pytest','-q',str(PROJECT/'tests')],cwd=PROJECT,check=True)
plan=json.loads((PROJECT/'research/JOURNAL_PLAN.json').read_text())
print(json.dumps(plan,indent=2))


## Execute the bounded plan
Images and source annotations remain in temporary storage. Output contains manifests, histories, checkpoints, numeric results and a private duplicate-review sheet. Do not redistribute source photographs.

In [ ]:
from src.journal import Campaign
OUTPUT=WORKING/'muon_yolo_journal'
try:
    campaign=Campaign(OUTPUT,plan,41880,preflight=False)
    campaign.run()
except Exception:
    OUTPUT.mkdir(parents=True,exist_ok=True)
    (OUTPUT/'notebook_failure.txt').write_text(traceback.format_exc(),encoding='utf-8')
    raise


In [ ]:
print((OUTPUT/'CAMPAIGN_FINDINGS.md').read_text())
print(json.dumps(json.loads((OUTPUT/'manifest.json').read_text()),indent=2))
